### Semantic Chunking
- It do the chunking based on similarity between sentances which helps the RAG model to gather the most create more relevant chunks

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity  

sample_text="""
he nightly data ingestion job failed at 02:15 AM due to a database connection timeout. 
The application logs indicate that the JDBC connection pool was exhausted, preventing new transactions from being created. 
As an immediate workaround, the operations team restarted the service, which temporarily resolved the issue.
However, long-term mitigation requires increasing the connection pool size and investigating slow-running queries that are not releasing connections properly.

"""

model = SentenceTransformer('all-MiniLM-L6-v2')

# Step 1: Split the text into sentences
sentences = [sentence.strip() for sentence in sample_text.split('.') if sentence.strip()]
for i, sentence in enumerate(sentences):
    print(f"Sentence {i+1}: {sentence}")
    
# Step 2: Generate embeddings for each sentence
embeddings = model.encode(sentences)

# Step 3: Compute cosine similarity between sentence embeddings
similarity_matrix = cosine_similarity(embeddings)

print("Sentence to Sentence Similarity Comparison")
for i in range(len(sentences)):
    for j in range(len(sentences)):
        print(f"Similarity between Sentence {i+1} and Sentence {j+1}: {similarity_matrix[i][j]:.4f}")

# Step 4: Print the similarity matrix
print("Cosine Similarity Matrix:")
print(np.round(similarity_matrix, 4))


Sentence 1: he nightly data ingestion job failed at 02:15 AM due to a database connection timeout
Sentence 2: The application logs indicate that the JDBC connection pool was exhausted, preventing new transactions from being created
Sentence 3: As an immediate workaround, the operations team restarted the service, which temporarily resolved the issue
Sentence 4: However, long-term mitigation requires increasing the connection pool size and investigating slow-running queries that are not releasing connections properly
Sentence to Sentence Similarity Comparison
Similarity between Sentence 1 and Sentence 1: 1.0000
Similarity between Sentence 1 and Sentence 2: 0.4311
Similarity between Sentence 1 and Sentence 3: 0.3451
Similarity between Sentence 1 and Sentence 4: 0.3418
Similarity between Sentence 2 and Sentence 1: 0.4311
Similarity between Sentence 2 and Sentence 2: 1.0000
Similarity between Sentence 2 and Sentence 3: 0.3735
Similarity between Sentence 2 and Sentence 4: 0.3559
Similarity 

In [6]:
# Merging similar sentences based on a threshold
threshold = 0.33
chunks=[]
current_chunk = [sentences[0]]
for i in range(1,len(sentences)):
    similarity=similarity_matrix[i-1][i];
    if similarity>threshold and i!=0:
       current_chunk.append(sentences[i])
    else:
       chunks.append(' '.join(current_chunk))
       current_chunk=[sentences[i]]

chunks.append(" ".join(current_chunk))
print("\nMerged Chunks based on Similarity Threshold:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")



Merged Chunks based on Similarity Threshold:
Chunk 1: he nightly data ingestion job failed at 02:15 AM due to a database connection timeout The application logs indicate that the JDBC connection pool was exhausted, preventing new transactions from being created As an immediate workaround, the operations team restarted the service, which temporarily resolved the issue
Chunk 2: However, long-term mitigation requires increasing the connection pool size and investigating slow-running queries that are not releasing connections properly


## RAG Pipeline with the Semantic Chunking

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from sklearn.metrics.pairwise import cosine_similarity
import os
from dotenv import load_dotenv
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
print("OPENROUTER_API_KEY:", OPENROUTER_API_KEY)

OPENROUTER_API_KEY: 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


In [8]:
class CustomSemanticChunking:
    def __init__(self,modelname='all-MiniLM-L6-v2',similarity_threshold=0.33):
        self.embedding_model = SentenceTransformer(modelname)
        self.similarity_threshold = similarity_threshold
        
    def split_text(self, text):
        sentences = [sentence.strip() for sentence in text.split('.') if sentence.strip()]
        embeddings = self.embedding_model.encode(sentences)
        similarity_matrix = cosine_similarity(embeddings)
        
        chunks=[]
        current_chunk = [sentences[0]]
        for i in range(1,len(sentences)):
            similarity=similarity_matrix[i-1][i];
            if similarity>self.similarity_threshold and i!=0:
               current_chunk.append(sentences[i])
            else:
               chunks.append(' '.join(current_chunk))
               current_chunk=[sentences[i]]
        chunks.append(" ".join(current_chunk))
        return chunks
        
    def split_documents(self, documents):
        all_chunks = []
        for doc in documents:
            chunks = self.split_text(doc.page_content)
            for chunk in chunks:
                all_chunks.append(Document(page_content=chunk, metadata=doc.metadata))
        return all_chunks
         

In [9]:
text_input="""


This is a sample text. It is used to demonstrate semantic chunking. 
Semantic chunking groups similar sentences together. This helps in better understanding of the text. 
The goal is to create meaningful chunks. Each chunk contains related information. 
This improves retrieval and processing of text data.
"""

input_docs = [Document(page_content=text_input, metadata={"source": "RAG Demo with Semantic Chunking"})]
print("Input Documents:")
for i,doc in enumerate(input_docs):
    print(f"Document {i+1}: {doc.page_content}")

Input Documents:
Document 1: 


This is a sample text. It is used to demonstrate semantic chunking. 
Semantic chunking groups similar sentences together. This helps in better understanding of the text. 
The goal is to create meaningful chunks. Each chunk contains related information. 
This improves retrieval and processing of text data.



In [10]:
chunker = CustomSemanticChunking(similarity_threshold=0.33)
chunked_docs = chunker.split_documents(input_docs)
print("\nChunked Documents:")
for i,doc in enumerate(chunked_docs):
    print(f"Chunk {i+1}: {doc.page_content}")


Chunked Documents:
Chunk 1: This is a sample text It is used to demonstrate semantic chunking Semantic chunking groups similar sentences together
Chunk 2: This helps in better understanding of the text
Chunk 3: The goal is to create meaningful chunks Each chunk contains related information This improves retrieval and processing of text data


In [ ]:
# Vector Store Creation with Chunked Documents
embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vector_store = Chroma.from_documents(documents=chunked_docs, embedding=embedding_model, collection_name="semantic_chunked_docs")
#retriver = vector_store.as_retriever(search_kwargs={"k": 1})

# Prompt and LLM Setup
prompt_template = """
You are an expert assistant. Use the following context to answer the question.
Context: {context}
Question: {question}
Answer:
"""
chat_prompt = ChatPromptTemplate.from_template(prompt_template)
llm=ChatOpenAI(
    model="xiaomi/mimo-v2-flash:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY   
)
chain = chat_prompt | llm | StrOutputParser()
# Example Query
query = "Describe semantic chunking"
relevant_chunks=vector_store.similarity_search(query, k=1)
context="\n".join([chunk.page_content for chunk in relevant_chunks])
response = chain.invoke({"context": context, "question": query})
print("\nQuery Response:")  
print(response)



Query Response:
Based on the context provided:

**Semantic chunking** is a method that groups similar sentences together.


### New way to use Retriver

In [ ]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import chain

@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)